

setup environment


In [1]:
!pip install umap-learn plotly pandas scikit-learn numpy scipy

First upload the zip of the github repo into the notebook (/content folder), or clone the repo from git


In [8]:

import os
if os.path.exists('/content/SNLP-Project-main'):
  pass
elif os.path.exists('/content/SNLP-Project-main.zip'):
  !unzip "/content/SNLP-Project-main.zip"
else:
  print('File SNLP-Project-main.zip not found.')



Archive:  /content/SNLP-Project-main.zip
e0856d7a1fa0260dca043a421bc9af30287807e5
   creating: SNLP-Project-main/
   creating: SNLP-Project-main/Dataset/
  inflating: SNLP-Project-main/Dataset/PoemDataset.csv  
  inflating: SNLP-Project-main/Dataset/PoetryFoundationData.csv  
  inflating: SNLP-Project-main/Dataset/birthyears.csv  
  inflating: SNLP-Project-main/Dataset/birthyears_cleaned.csv  
  inflating: SNLP-Project-main/Dataset/birthyears_wiki.csv  
  inflating: SNLP-Project-main/Dataset/dataset_merge.ipynb  
   creating: SNLP-Project-main/Dataset/figures/
  inflating: SNLP-Project-main/Dataset/figures/fig_emotion_stats.png  
  inflating: SNLP-Project-main/Dataset/figures/fig_period_stats.png  
  inflating: SNLP-Project-main/Dataset/figures/fig_poet_stats.png  
  inflating: SNLP-Project-main/Dataset/poem_data_with_birth.csv  
  inflating: SNLP-Project-main/Dataset/scrape.ipynb  
   creating: SNLP-Project-main/Embeddings/
  inflating: SNLP-Project-main/Embeddings/add_format_info.ipy

In [9]:


import numpy as np
import pandas as pd
import ast
import logging
import shutil
from pathlib import Path
from scipy.sparse import load_npz, issparse
import plotly.graph_objects as go
import plotly.colors
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.pipeline import make_pipeline

ROOT_DIR = "/content/SNLP-Project-main"
os.chdir(ROOT_DIR)

CONFIG = {
    "pca_components": 50,
    "svd_components": 50,
    "umap_n_neighbors": 15,
    "umap_min_dist": 0.1,
    "umap_metric": "cosine",
    "tsne_perplexity": 30,
    "snippet_length": 80,
    "top_poets_n": 20,
    "plot_point_size": 5,
    "plot_opacity": 0.6,
    "col_title":    "Title",
    "col_author":   "Poet",
    "col_poem":     "Poem",
    "col_tags":     "Tags",
    "col_emotion":  "Emotion",
    "col_period":   "Period",
    "col_format":   "Format",
    "col_cluster_id":  "cluster_id",
    "col_parsed_tags": "parsed_top_level_tags",
    "col_snippet":      "Snippet",
    "col_plot_poet":    "Plot_Poet",
    "col_plot_emotion": "Plot_Emotion",
    "col_plot_format":  "Plot_Format",
    "col_plot_cluster":  "Plot_Cluster",
    "cache_dir": "cache",
    "output_dir": "plots",
}
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("viz")

#Plot formatting
_PALETTE_PERIOD  = ['rgb(68, 1, 84)', 'rgb(64, 66, 134)', 'rgb(42, 120, 142)', 'rgb(40, 168, 131)', 'rgb(124, 209, 79)', 'rgb(253, 231, 37)']
_PALETTE_EMOTION = plotly.colors.qualitative.Bold
_PALETTE_POET    = plotly.colors.qualitative.Vivid
_PALETTE_CLUSTER = plotly.colors.qualitative.Dark24
_PALETTE_FORMAT  = plotly.colors.qualitative.Pastel
_CHRONOLOGICAL_PERIODS = ["Pre-1550", "1550-1780", "1781-1900", "1901-1940", "1941-1970", "1971-Present"]
HIGHLIGHT_COLOR = "#E63946"
BACKGROUND_COLOR = "rgba(200, 200, 200, 0.3)"
TARGET_TAGS = ["Religion", "Mythology & Folklore", "Nature", "Love", "Social Commentaries"]
_GRAY_OTHER   = "rgba(180, 180, 180, 0.25)"
_GRAY_UNKNOWN = "rgba(160, 160, 160, 0.30)"



Data loading and merging functions



In [10]:

def load_vectors(path: str) -> np.ndarray:
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f"Vector file not found: {path}")
    log.info(f"Loading vectors: {path}")
    if p.suffix == ".npz":
        mat = load_npz(path)
        log.info(f"  Sparse matrix: {mat.shape}, ~{mat.data.nbytes / 1e6:.1f} MB")
        return mat
    else:
        data = np.load(path, allow_pickle=True)
        if isinstance(data, np.lib.npyio.NpzFile):
            key = "vectors" if "vectors" in data else list(data.keys())[0]
            mat = data[key]
        else:
            mat = data
        log.info(f"  Dense matrix: {mat.shape}")
        return mat

def load_metadata(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f"Metadata file not found: {path}")
    df = pd.read_csv(path)
    log.info(f"Loaded metadata: {len(df)} rows")
    return df

def load_cluster_metadata(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f"Cluster metadata not found: {path}")
    cluster_df = pd.read_csv(path)
    col_cluster, col_parsed = CONFIG["col_cluster_id"], CONFIG["col_parsed_tags"]
    keep = [c for c in [col_cluster, col_parsed] if c in cluster_df.columns]
    return cluster_df[keep].copy()

def merge_cluster_data(df: pd.DataFrame, cluster_df: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([df.reset_index(drop=True), cluster_df.reset_index(drop=True)], axis=1)



Data cleaning and formatting for visualization


In [11]:
"""Data cleaning and formatting for visualization"""
def engineer_features(df: pd.DataFrame, has_cluster: bool = False, available_tasks: list = None) -> pd.DataFrame:
    df = df.copy()
    available_tasks = available_tasks or []

    col_poem = CONFIG["col_poem"]
    snippet_len = CONFIG["snippet_length"]
    df[CONFIG["col_snippet"]] = df[col_poem].fillna("").astype(str).str[:snippet_len] if col_poem in df.columns else ""

    col_emotion = CONFIG["col_emotion"]
    df[CONFIG["col_plot_emotion"]] = df[col_emotion].fillna("Unknown").replace("", "Unknown") if col_emotion in df.columns else "Unknown"

    col_author, top_n = CONFIG["col_author"], CONFIG["top_poets_n"]
    if col_author in df.columns:
        top_poets = df[col_author].value_counts().nlargest(top_n).index
        df[CONFIG["col_plot_poet"]] = df[col_author].where(df[col_author].isin(top_poets), other="Other")
    else: df[CONFIG["col_plot_poet"]] = "Unknown"

    col_period = CONFIG["col_period"]
    df[col_period] = df[col_period].fillna("Unknown") if col_period in df.columns else "Unknown"

    col_format = CONFIG["col_format"]
    df[CONFIG["col_plot_format"]] = df[col_format].fillna("Unknown") if col_format in df.columns else "Unknown"

    col_tags = CONFIG["col_tags"]
    df[col_tags] = df[col_tags].fillna("") if col_tags in df.columns else ""

    col_cluster, col_parsed = CONFIG["col_cluster_id"], CONFIG["col_parsed_tags"]
    if has_cluster:
        df[CONFIG["col_plot_cluster"]] = df[col_cluster].fillna(-1).astype(int).astype(str) if col_cluster in df.columns else "N/A"
        if col_parsed in df.columns:
            def safe_parse(val):
                try: return ast.literal_eval(val) if isinstance(val, str) else list(val)
                except: return []
            df[col_parsed] = df[col_parsed].fillna("[]").apply(safe_parse)
        else: df[col_parsed] = [[]] * len(df)
    else:
        df[CONFIG["col_plot_cluster"]] = "N/A"
        df[col_parsed] = [[]] * len(df)

    for task in available_tasks:
        task_col = f"cluster_id_{task}"
        if task_col in df.columns:
            df[f"Plot_Cluster_{task}"] = df[task_col].fillna(-1).astype(int).astype(str)

    return df



Dimensionality reduction functions


In [12]:
"""Dimensionality reduction functions"""
def _save_cache(array: np.ndarray, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, array)

def reduce_to_2d(vectors, model_name: str, algorithm: str) -> np.ndarray:
    cache_path = Path(CONFIG["cache_dir"]) / f"{model_name}_{algorithm}_2d.npy"
    if cache_path.exists():
        return np.load(cache_path)
    #if tf-ifd, do svd and normalize
    if issparse(vectors):
        svd = make_pipeline(TruncatedSVD(n_components=CONFIG["svd_components"], random_state=42), Normalizer(copy=False))
        intermediate = svd.fit_transform(vectors)
    else: #if neural net, do PCA
        scaled = StandardScaler().fit_transform(vectors)
        n_comp = min(CONFIG["pca_components"], scaled.shape[1])
        pca = PCA(n_components=n_comp, random_state=42)
        intermediate = pca.fit_transform(scaled)
    #umap coordinates
    if algorithm == "umap":
        reducer = umap.UMAP(n_components=2, n_neighbors=CONFIG["umap_n_neighbors"], min_dist=CONFIG["umap_min_dist"], metric=CONFIG["umap_metric"], random_state=42, verbose=True)
        coords_2d = reducer.fit_transform(intermediate)
    #t-SNE coordinates
    elif algorithm == "tsne":
        reducer = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto", verbose=1)
        coords_2d = reducer.fit_transform(intermediate)
    _save_cache(coords_2d, cache_path)
    return coords_2d



Visualization functions


In [13]:
"""Visualization functions"""
def _assign_colors(categories: list[str], palette: list[str], special: dict[str, str] | None = None) -> dict[str, str]:
    special = special or {}
    normal = [c for c in categories if c not in special]
    cmap = {cat: palette[i % len(palette)] for i, cat in enumerate(normal)}
    cmap.update(special)
    return cmap

def _build_customdata(df: pd.DataFrame, has_cluster: bool, available_tasks: list) -> np.ndarray:
    cols = [CONFIG["col_title"], CONFIG["col_author"], CONFIG["col_period"], CONFIG["col_plot_emotion"], CONFIG["col_tags"], CONFIG["col_snippet"], CONFIG["col_plot_format"]]
    if has_cluster: cols.extend([CONFIG["col_parsed_tags"], CONFIG["col_plot_cluster"]])
    for task in available_tasks:
        cols.append(f"Plot_Cluster_{task}")
    for c in cols:
        if c not in df.columns: df[c] = ""
    return df[cols].fillna("").values

def _create_traces_for_column(x, y, df, color_col, color_map, customdata, visible, hover) -> list:
    traces = []
    categories = sorted(df[color_col].unique(), key=lambda c: (1, c) if c in ("Other", "Unknown", "N/A") else (0, c))
    for cat in categories:
        mask = df[color_col] == cat
        traces.append(go.Scatter(x=x[mask], y=y[mask], mode="markers", name=str(cat), marker=dict(size=CONFIG["plot_point_size"], opacity=CONFIG["plot_opacity"], color=color_map.get(cat, "#888888")), customdata=customdata[mask], hovertemplate=hover, visible=visible, showlegend=True))
    return traces

def build_figure(coords_2d: np.ndarray, df: pd.DataFrame, model_name: str, algorithm: str, has_cluster: bool = False, available_tasks: list = None) -> go.Figure:
    available_tasks = available_tasks or []
    x, y = coords_2d[:, 0], coords_2d[:, 1]
    customdata = _build_customdata(df, has_cluster, available_tasks)

    hover = "<b>%{customdata[0]}</b><br>By: %{customdata[1]}<br>Period: %{customdata[2]}<br>Emotion: %{customdata[3]}<br>Format: %{customdata[6]}<br>Tags: %{customdata[4]}<br>"
    cd_idx = 7
    if has_cluster:
        hover += f"Topic Tags: %{{customdata[{cd_idx}]}}<br>Cluster: %{{customdata[{cd_idx+1}]}}<br>"
        cd_idx += 2
    for task in available_tasks:
        hover += f"{task} Cluster: %{{customdata[{cd_idx}]}}<br>"
        cd_idx += 1

    hover += "<hr><i>\"%{customdata[5]}...\"</i><extra></extra>"

    col_period, col_emotion, col_poet, col_format = CONFIG["col_period"], CONFIG["col_plot_emotion"], CONFIG["col_plot_poet"], CONFIG["col_plot_format"]

    raw_periods = set(df[col_period].unique())
    period_cats = [p for p in _CHRONOLOGICAL_PERIODS if p in raw_periods] + sorted([p for p in raw_periods if p not in _CHRONOLOGICAL_PERIODS])
    emotion_cats, poet_cats, format_cats = list(df[col_emotion].unique()), list(df[col_poet].unique()), list(df[col_format].unique())

    map_period  = _assign_colors(period_cats, _PALETTE_PERIOD)
    map_emotion = _assign_colors(emotion_cats, _PALETTE_EMOTION, {"Unknown": _GRAY_UNKNOWN})
    map_poet    = _assign_colors(poet_cats, _PALETTE_POET, {"Other": _GRAY_OTHER})
    map_format  = _assign_colors(format_cats, _PALETTE_FORMAT, {"Unknown": _GRAY_UNKNOWN})

    traces_period  = _create_traces_for_column(x, y, df, col_period, map_period, customdata, True, hover)
    traces_emotion = _create_traces_for_column(x, y, df, col_emotion, map_emotion, customdata, False, hover)
    traces_poet    = _create_traces_for_column(x, y, df, col_poet, map_poet, customdata, False, hover)
    traces_format  = _create_traces_for_column(x, y, df, col_format, map_format, customdata, False, hover)

    trace_groups = [traces_period, traces_emotion, traces_poet, traces_format]

    if has_cluster:
        col_cluster = CONFIG["col_plot_cluster"]
        traces_cluster = _create_traces_for_column(x, y, df, col_cluster, _assign_colors(list(df[col_cluster].unique()), _PALETTE_CLUSTER, {"N/A": _GRAY_UNKNOWN}), customdata, False, hover)
        trace_groups.append(traces_cluster)

        traces_highlight = []
        for tag in TARGET_TAGS:
            colors = [HIGHLIGHT_COLOR if tag in tags else BACKGROUND_COLOR for tags in df[CONFIG["col_parsed_tags"]]]
            sizes = [6 if tag in tags else 3 for tags in df[CONFIG["col_parsed_tags"]]]
            traces_highlight.append(go.Scatter(x=x, y=y, mode="markers", name=f"Highlight: {tag}", marker=dict(color=colors, size=sizes, opacity=1.0), customdata=customdata, hovertemplate=hover, visible=False, showlegend=False))
        trace_groups.append(traces_highlight)

    task_trace_groups = []
    for task in available_tasks:
        col_name = f"Plot_Cluster_{task}"
        t_traces = _create_traces_for_column(x, y, df, col_name, _assign_colors(list(df[col_name].unique()), _PALETTE_CLUSTER, {"N/A": _GRAY_UNKNOWN, "-1": _GRAY_UNKNOWN}), customdata, False, hover)
        trace_groups.append(t_traces)
        task_trace_groups.append(t_traces)

    all_traces = []
    for group in trace_groups:
        all_traces.extend(group)

    n_groups = len(trace_groups)
    group_lengths = [len(g) for g in trace_groups]

    def get_visibility(active_idx):
        vis = []
        for i, length in enumerate(group_lengths):
            vis.extend([i == active_idx] * length)
        return vis

    buttons = [
        dict(label="🕰️  Color by Period", method="update", args=[{"visible": get_visibility(0)}, {"legend.title.text": "Period"}]),
        dict(label="💡  Color by Emotion", method="update", args=[{"visible": get_visibility(1)}, {"legend.title.text": "Emotion"}]),
        dict(label="✍️  Color by Top 20 Poets", method="update", args=[{"visible": get_visibility(2)}, {"legend.title.text": "Poet"}]),
        dict(label="📐  Color by Format", method="update", args=[{"visible": get_visibility(3)}, {"legend.title.text": "Format"}]),
    ]

    group_idx = 4
    if has_cluster:
        buttons.append(dict(label="🔬  Color by K-Means Cluster", method="update", args=[{"visible": get_visibility(group_idx)}, {"legend.title.text": "K-Means Cluster"}]))
        group_idx += 1

        # Add highlight buttons
        hl_group_len = group_lengths[group_idx]
        for i, tag in enumerate(TARGET_TAGS):
            vis_tag = []
            for j, length in enumerate(group_lengths):
                if j == group_idx:
                    # Only the specific highlight trace is visible
                    vis_arr = [False] * length
                    vis_arr[i] = True
                    vis_tag.extend(vis_arr)
                else:
                    vis_tag.extend([False] * length)
            buttons.append(dict(label=f"🌟 Highlight: {tag}", method="update", args=[{"visible": vis_tag}, {"legend.title.text": f"Tag: {tag}"}]))
        group_idx += 1

    for i, task in enumerate(available_tasks):
        buttons.append(dict(label=f"🔬  Cluster: {task}", method="update", args=[{"visible": get_visibility(group_idx + i)}, {"legend.title.text": f"Cluster: {task}"}]))

    fig = go.Figure(data=all_traces)
    fig.update_layout(title=dict(text=f"{algorithm.upper()} - {model_name}"), template="plotly_dark", legend=dict(itemsizing="constant"), xaxis=dict(showgrid=False, zeroline=False, showticklabels=False), yaxis=dict(showgrid=False, zeroline=False, showticklabels=False), updatemenus=[dict(type="dropdown", direction="down", x=0.01, xanchor="left", y=1.12, yanchor="top", showactive=True, active=0, buttons=buttons)], margin=dict(t=100, l=40, r=40, b=40))
    return fig



### UMAP and tSNE Parameters


In [14]:

# Number of dimensions to reduce to before UMAP/t-SNE (10 to 100)
N_COMPONENTS = 50

TSNE_PERPLEXITY = 30

# Balances local vs global structure (5 to 100)
UMAP_NEIGHBORS = 15
# Controls cluster clumpiness (0.01 to 0.5)
UMAP_MIN_DIST = 0.1

import shutil
from pathlib import Path
if Path("cache").exists():  # Note: assuming cache_dir is "cache"
    shutil.rmtree("cache")

# Update Config
CONFIG["pca_components"] = N_COMPONENTS
CONFIG["svd_components"] = N_COMPONENTS
CONFIG["umap_n_neighbors"] = UMAP_NEIGHBORS
CONFIG["umap_min_dist"]    = UMAP_MIN_DIST
CONFIG["tsne_perplexity"] = TSNE_PERPLEXITY

print(f"Applying PCA={N_COMPONENTS} | UMAP_Neighbors={UMAP_NEIGHBORS} | tSNE_Perplexity={TSNE_PERPLEXITY}")



Applying PCA=50 | UMAP_Neighbors=15 | tSNE_Perplexity=30


Main/execution


In [16]:
"""Main/execution"""
MODELS = {
    "d2v_poem_d100_w5_dbow": "d2v_poem_d100_w5_dbow_tag_clusters.csv",
    "d2v_line_d100_w5_dbow": "d2v_line_d100_w5_dbow_tag_clusters.csv",
    "w2v_d100_w5_sg_idf":    "w2v_d100_w5_sg_idf_tag_clusters.csv",
    "baseline":                   None,
}

embeddings_dir = Path("embeddings") if Path("embeddings").exists() else Path("/content/SNLP-Project-main/Embeddings/filtered_embeddings")
metadata_dir   = Path("metadata") if Path("metadata").exists() else Path("/content/SNLP-Project-main/Embeddings/results/cluster_analysis")
output_dir     = Path("plots")
output_dir.mkdir(exist_ok=True)
base_metadata  = Path("filtered_poems.csv") if Path("filtered_poems.csv").exists() else Path("/content/SNLP-Project-main/Embeddings/filtered_poems.csv")

# For local project we override to known locations
embeddings_dir = Path("embeddings/filtered_embeddings") if Path("embeddings/filtered_embeddings").exists() else embeddings_dir
if Path("metadata/filtered_poems.csv").exists():
    base_metadata = Path("metadata/filtered_poems.csv")

for model_stem, cluster_file in MODELS.items():
    vec_path = embeddings_dir / f"{model_stem}.npy"
    if not vec_path.exists():
        vec_path = Path("metadata") / f"{model_stem}.npy" # fallbacks
        if not vec_path.exists():
            log.warning(f"Embedding file not found, skipping: {vec_path}")
            continue

    print(f"PROCESSING MODEL: {model_stem}")
    cluster_path = str(metadata_dir / cluster_file) if cluster_file else None
    has_cluster = False
    if cluster_path and Path(cluster_path).exists():
        has_cluster = True

    tasks = ["Emotion", "Format", "Period", "PoetTop20"]
    available_tasks = []
    task_df_list = []
    for task in tasks:
        tc_file = metadata_dir / "cluster_assignments" / f"{model_stem}_{task}_clusters.csv"
        if not tc_file.exists():
            tc_file = Path("metadata") / f"{model_stem}_{task}_clusters.csv"

        if tc_file.exists():
            tc_df = pd.read_csv(tc_file)
            if "cluster_id" in tc_df.columns:
                task_df_list.append((task, tc_df["cluster_id"]))
                available_tasks.append(task)

    # Process Data
    vectors = load_vectors(str(vec_path))
    df = load_metadata(base_metadata)

    if has_cluster:
        cluster_df = load_cluster_metadata(cluster_path)
        df = merge_cluster_data(df, cluster_df)

    for task, col_data in task_df_list:
        df[f"cluster_id_{task}"] = col_data

    df = engineer_features(df, has_cluster, available_tasks)
    for algorithm in ["umap", "tsne"]:

      # Calculate cordinates
      coords_2d = reduce_to_2d(vectors, model_stem, algorithm)

      #  Interactive HTML
      interactive_fig = build_figure(coords_2d, df, model_stem, algorithm, has_cluster, available_tasks)
      html_out = str(output_dir / f"{algorithm}_visualization_{model_stem}.html")
      interactive_fig.write_html(html_out, include_plotlyjs="cdn")

      # Define the attributes
      visualizations = [
          (CONFIG["col_period"], "Time Period", "viridis"),
          (CONFIG["col_plot_emotion"], "Emotion", "tab10"),
          (CONFIG["col_plot_poet"], "Top 20 Poets", "Set3"),
          (CONFIG["col_plot_format"], "Format", "Set2")
      ]
      if has_cluster:
          visualizations.append((CONFIG["col_plot_cluster"], "K-Means Cluster", "Dark2"))

      for task in available_tasks:
          visualizations.append((f"Plot_Cluster_{task}", f"{task} Cluster", "Dark2"))

      # Draw static plots
      for col_name, title, palette in visualizations:
          plt.figure(figsize=(9, 6))
          sns.scatterplot(
              x=coords_2d[:, 0], y=coords_2d[:, 1],
              hue=df[col_name], palette=palette, s=12, alpha=0.6, edgecolor=None
          )
          plt.title(f"{algorithm.upper()} Projection: {model_stem}\n(By {title})", pad=15)
          plt.xticks([]); plt.yticks([]); plt.xlabel(""); plt.ylabel("")
          plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title=title, fontsize='small')
          plt.tight_layout()

          plot_path = output_dir / f"{algorithm}_{model_stem}_{col_name}.png"
          plt.savefig(plot_path, bbox_inches='tight')
          plt.close() # clear memory instead of showing popups


PROCESSING MODEL: d2v_poem_d100_w5_dbow
PROCESSING MODEL: d2v_line_d100_w5_dbow
PROCESSING MODEL: w2v_d100_w5_sg_idf
PROCESSING MODEL: baseline
